In [ ]:
#| default_exp train_flow

# Flow Matching Generative Model

Trains a flow matching model on pre-encoded (optionally PCA-reduced) embeddings.
Source distribution is N(0,I); target is the embedding distribution.
Uses RK4 integration and optional time warping at inference.

In [ ]:
#| export
import gc
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
from midi_rae.data import EmbeddingDataset

In [ ]:
#| export
class VelocityNet(nn.Module):
    """MLP velocity field for flow matching.  Input: [x, (x_self_cond,) t_emb], output: dx/dt.
    Hidden layers use residual (skip) connections.
    t_dim: sinusoidal time embedding dim (replaces bare scalar t).
    self_condition: if True, also accepts x_self_cond (predicted x1 from prior pass); zeros when absent."""
    def __init__(self, input_dim, h_dim=256, n_layers=3, self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.t_dim = t_dim
        net_in = input_dim * 2 + t_dim if self_condition else input_dim + t_dim
        self.fc_in  = nn.Linear(net_in, h_dim)
        self.hidden = nn.ModuleList([nn.Linear(h_dim, h_dim) for _ in range(n_layers - 1)])
        self.fc_out = nn.Linear(h_dim, input_dim)

    def forward(self, x, t, x_self_cond=None):
        if t.dim() > 1: t = t.squeeze(-1)
        elif t.dim() == 0: t = t.unsqueeze(0).expand(x.size(0))
        t_emb = sinusoidal_time_emb(t, self.t_dim)          # [B, t_dim]
        if self.self_condition:
            sc = x_self_cond if x_self_cond is not None else torch.zeros_like(x)
            inp = torch.cat([x, sc, t_emb], dim=1)
        else:
            inp = torch.cat([x, t_emb], dim=1)
        h = F.gelu(self.fc_in(inp))
        for layer in self.hidden:
            h = F.gelu(layer(h)) + h
        return self.fc_out(h)


In [ ]:
#| export
import math

def sinusoidal_time_emb(t, dim=64):
    """Sinusoidal time embedding (à la DDPM/DiT).  t: [B] or [B,1] → [B, dim].
    Gives the model a rich multi-frequency view of t instead of a bare scalar."""
    if t.dim() > 1: t = t.squeeze(-1)
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, dtype=torch.float32, device=t.device) / (half - 1))
    x = t.float().unsqueeze(1) * freqs.unsqueeze(0)   # [B, half]
    return torch.cat([x.sin(), x.cos()], dim=-1)       # [B, dim]


In [ ]:
#| export
class PerLevelFlowModel(nn.Module):
    """One VelocityNet per embedding level; each level's slice is routed to its own net.
    Has the same forward(x, t, x_self_cond=None) interface as VelocityNet.
    level_dims: list of ints, e.g. [20, 80, 320, 1280] from dataset.level_dims
    self_condition / t_dim: passed through to each VelocityNet.
    """
    def __init__(self, level_dims, h_dim=256, n_layers=4, self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.level_dims = level_dims
        self.nets = nn.ModuleList([VelocityNet(d, h_dim, n_layers, self_condition=self_condition, t_dim=t_dim)
                                   for d in level_dims])

    def forward(self, x, t, x_self_cond=None):
        outs, offset = [], 0
        for net, d in zip(self.nets, self.level_dims):
            sc_slice = x_self_cond[:, offset:offset+d] if x_self_cond is not None else None
            outs.append(net(x[:, offset:offset+d], t, sc_slice))
            offset += d
        return torch.cat(outs, dim=1)


In [ ]:
#| export
class CrossLevelFlowModel(nn.Module):
    """Flow model with cross-level attention for joint velocity prediction.
    Each level is projected to h_dim, t is embedded sinusoidally and added to every token,
    a small transformer cross-attends across all levels (so L3 attends to L0-L2 without
    cascading), per-level residual MLPs refine, per-level heads decode velocity.
    Sequence length = n_levels (typically 4) so attention cost is negligible.
    Uses norm_first=True (pre-LN) for training stability.
    self_condition: same 50%-dropout self-conditioning as VelocityNet.
    t_dim: sinusoidal time embedding dim, projected to h_dim and added to each level token.
    """
    def __init__(self, level_dims, h_dim=512, n_layers=4, n_attn_layers=2, n_heads=8,
                 self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.level_dims = level_dims
        self.t_dim = t_dim
        in_mul = 2 if self_condition else 1
        # Per-level input projections: [x_level, (x_sc)] → h_dim  (t added separately)
        self.level_in  = nn.ModuleList([nn.Linear(d * in_mul, h_dim) for d in level_dims])
        # Time embedding: sinusoidal t_dim → h_dim (added to every level token)
        self.t_proj = nn.Sequential(nn.Linear(t_dim, h_dim), nn.SiLU(), nn.Linear(h_dim, h_dim))
        # Cross-level transformer (seq_len = n_levels ≈ 4; norm_first=True for stability)
        enc_layer = nn.TransformerEncoderLayer(h_dim, n_heads, dim_feedforward=h_dim * 4,
                                               batch_first=True, dropout=0.0, norm_first=True)
        self.cross_attn = nn.TransformerEncoder(enc_layer, num_layers=n_attn_layers)
        # Per-level residual MLPs (operate in h_dim space)
        self.level_mlp = nn.ModuleList([
            nn.ModuleList([nn.Linear(h_dim, h_dim) for _ in range(n_layers - 1)])
            for _ in level_dims])
        # Per-level output heads → velocity
        self.level_out = nn.ModuleList([nn.Linear(h_dim, d) for d in level_dims])

    def forward(self, x, t, x_self_cond=None):
        if t.dim() > 1: t = t.squeeze(-1)
        elif t.dim() == 0: t = t.unsqueeze(0).expand(x.size(0))
        t_emb = self.t_proj(sinusoidal_time_emb(t, self.t_dim))   # [B, h_dim]
        # Build per-level tokens
        tokens, offset = [], 0
        for proj, d in zip(self.level_in, self.level_dims):
            xd = x[:, offset:offset+d]
            if self.self_condition:
                sc = x_self_cond[:, offset:offset+d] if x_self_cond is not None else torch.zeros_like(xd)
                inp = torch.cat([xd, sc], dim=1)
            else:
                inp = xd
            tokens.append(F.gelu(proj(inp)) + t_emb)   # add time additively
            offset += d
        # Cross-attend across levels: [B, n_levels, h_dim]
        tokens = self.cross_attn(torch.stack(tokens, dim=1))
        # Per-level residual MLP + output head
        outs = []
        for mlp_layers, out_proj, tok in zip(self.level_mlp, self.level_out, tokens.unbind(1)):
            h = tok
            for layer in mlp_layers:
                h = F.gelu(layer(h)) + h
            outs.append(out_proj(h))
        return torch.cat(outs, dim=1)


In [ ]:
#| export
class FiLM(nn.Module):
    """Feature-wise Linear Modulation with pre-norm (AdaLN style).

    Applies adaptive LayerNorm: output = (1 + γ(cond)) * LN(x) + β(cond).
    Weights initialised to zero so the module starts as a plain LayerNorm
    (identity modulation), giving stable early training.

    Args:
        cond_dim: dimensionality of the conditioning vector.
        feat_dim: dimensionality of the features to modulate.
    """
    def __init__(self, cond_dim, feat_dim):
        super().__init__()
        self.norm  = nn.LayerNorm(feat_dim)
        self.gamma = nn.Linear(cond_dim, feat_dim)
        self.beta  = nn.Linear(cond_dim, feat_dim)
        # Init: γ≈0, β≈0  →  output ≈ LN(x) at the start of training
        nn.init.zeros_(self.gamma.weight); nn.init.zeros_(self.gamma.bias)
        nn.init.zeros_(self.beta.weight);  nn.init.zeros_(self.beta.bias)

    def forward(self, x, cond):
        return (1 + self.gamma(cond)) * self.norm(x) + self.beta(cond)


class ConditionalFineFlowModel(nn.Module):
    """Second-stage flow model: generates fine-level embeddings (L4/L5)
    conditioned on coarse-level embeddings (L0-L3) from a first-stage flow.

    Design:
      - Input LayerNorm on each level before projection (normalises scale
        differences between levels; L4/L5 dims are tiny vs L0-L3).
      - All level tokens (cond + target) are projected to h_dim and jointly
        processed by a pre-norm TransformerEncoder so target tokens can freely
        attend to coarse context.
      - A pooled summary of the attended coarse tokens drives FiLM layers
        that modulate each target-level MLP block.  FiLM = AdaLN: the cond
        signal controls *scale and shift*, not just additive bias.
      - Time embedding: same sinusoidal + two-layer MLP as CrossLevelFlowModel.
        **The caller must pass the same `t` to both models during sampling**
        (enforced in generate_samples_conditional via a shared timestep grid).
      - Output heads exist only for target levels; coarse tokens have no
        velocity output and are detached from the loss.

    Args:
        cond_dims:    list of flattened dims for conditioning levels (L0-L3).
                      Should be PCA-compressed dims to match first-stage output.
        target_dims:  list of flattened dims for target levels (L4, L5).
        h_dim:        internal hidden dimension.
        n_layers:     total depth: 1 attention block + (n_layers-1) FiLM-MLP blocks.
        n_attn_layers: transformer encoder depth (seq_len = n_cond+n_target ≈ 6).
        n_heads:      attention heads (h_dim must be divisible by n_heads).
        t_dim:        sinusoidal time embedding dim (match first-stage model).
    """
    def __init__(self, cond_dims, target_dims, h_dim=256, n_layers=4,
                 n_attn_layers=2, n_heads=8, t_dim=64):
        super().__init__()
        self.cond_dims   = list(cond_dims)
        self.target_dims = list(target_dims)
        self.t_dim = t_dim
        n_cond = len(cond_dims)

        # --- Input normalisation (pre-projection LayerNorm per level) ---
        self.cond_norms   = nn.ModuleList([nn.LayerNorm(d) for d in cond_dims])
        self.target_norms = nn.ModuleList([nn.LayerNorm(d) for d in target_dims])

        # --- Per-level input projections → h_dim ---
        self.cond_in   = nn.ModuleList([nn.Linear(d, h_dim) for d in cond_dims])
        self.target_in = nn.ModuleList([nn.Linear(d, h_dim) for d in target_dims])

        # --- Time embedding: sinusoidal t_dim → h_dim (same as CrossLevelFlowModel) ---
        self.t_proj = nn.Sequential(
            nn.Linear(t_dim, h_dim), nn.SiLU(), nn.Linear(h_dim, h_dim))

        # --- Joint self-attention over all tokens (pre-norm for stability) ---
        enc_layer = nn.TransformerEncoderLayer(
            h_dim, n_heads, dim_feedforward=h_dim * 4,
            batch_first=True, dropout=0.0, norm_first=True)
        self.attn = nn.TransformerEncoder(enc_layer, num_layers=n_attn_layers)

        # --- Condition summary → FiLM input ---
        # Flatten attended coarse tokens and project to a single h_dim vector.
        self.cond_pool = nn.Sequential(
            nn.Linear(n_cond * h_dim, h_dim), nn.SiLU())

        # --- FiLM-modulated residual MLP blocks (one per depth step, per target level) ---
        self.film_layers = nn.ModuleList([
            nn.ModuleList([FiLM(h_dim, h_dim) for _ in target_dims])
            for _ in range(n_layers - 1)])
        self.mlp_layers = nn.ModuleList([
            nn.ModuleList([nn.Linear(h_dim, h_dim) for _ in target_dims])
            for _ in range(n_layers - 1)])

        # --- Output heads: velocity for target levels only ---
        self.target_out = nn.ModuleList([nn.Linear(h_dim, d) for d in target_dims])

    def forward(self, x_target, t, x_cond):
        """
        x_target : [B, sum(target_dims)]  noisy fine-level embeddings
        t         : [B] or [B,1]          timestep — must equal the t used by
                                           the first-stage model in the same step
        x_cond    : [B, sum(cond_dims)]   coarse embeddings from first-stage flow
                    (PCA-roundtripped during training to match inference distribution)
        Returns   : [B, sum(target_dims)] predicted velocity for target levels only
        """
        if t.dim() > 1: t = t.squeeze(-1)
        elif t.dim() == 0: t = t.unsqueeze(0).expand(x_target.size(0))
        assert t.min() >= 0.0 and t.max() <= 1.0, \
            "t must be in [0,1] — pass the same timestep used by the first-stage flow"

        t_emb = self.t_proj(sinusoidal_time_emb(t, self.t_dim))  # [B, h_dim]

        # Project condition levels (pre-norm → linear → GELU + time)
        cond_tokens, offset = [], 0
        for norm, proj, d in zip(self.cond_norms, self.cond_in, self.cond_dims):
            cond_tokens.append(F.gelu(proj(norm(x_cond[:, offset:offset+d]))) + t_emb)
            offset += d

        # Project target levels (pre-norm → linear → GELU + time)
        target_tokens, offset = [], 0
        for norm, proj, d in zip(self.target_norms, self.target_in, self.target_dims):
            target_tokens.append(F.gelu(proj(norm(x_target[:, offset:offset+d]))) + t_emb)
            offset += d

        # Joint self-attention: [B, n_cond + n_target, h_dim]
        all_tokens = self.attn(torch.stack(cond_tokens + target_tokens, dim=1))
        n_cond = len(self.cond_dims)
        cond_out    = all_tokens[:, :n_cond, :]
        target_toks = list(all_tokens[:, n_cond:, :].unbind(1))

        # Pool attended coarse tokens → FiLM conditioning signal
        film_cond = self.cond_pool(cond_out.flatten(1))  # [B, h_dim]

        # FiLM-modulated residual MLPs for each target level
        for film_row, mlp_row in zip(self.film_layers, self.mlp_layers):
            target_toks = [
                F.gelu(mlp(film(tok, film_cond))) + tok
                for tok, film, mlp in zip(target_toks, film_row, mlp_row)]

        # Output heads → velocity (target levels only)
        return torch.cat([out(tok) for out, tok in zip(self.target_out, target_toks)], dim=1)

In [ ]:
#| export
def warp_time(t, s=0.5):
    """Parametric time warping (Scott H. Hawley, 'Flow With What You Know', ICLR 2025).
    s=1 → linear; s<1 → slower near middle; s=1.5 ≈ cosine schedule.
    Works on scalar, 1-D or 2-D tensors."""
    return 4*(1-s)*t**3 + 6*(s-1)*t**2 + (3-2*s)*t

In [ ]:
#| export
@torch.no_grad()
def rk4_step(model, y, t, dt):
    """4th-order Runge-Kutta step for the learned velocity field."""
    t_  = torch.full((y.size(0), 1), t, device=y.device, dtype=y.dtype)
    k1 = model(y,             t_)
    k2 = model(y + dt*k1/2,   t_ + dt/2)
    k3 = model(y + dt*k2/2,   t_ + dt/2)
    k4 = model(y + dt*k3,     t_ + dt)
    return y + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

@torch.no_grad()
def euler_step(model, y, t, dt):
    t_ = torch.full((y.size(0), 1), t, device=y.device, dtype=y.dtype)
    return y + model(y, t_) * dt

In [ ]:
#| export
def sample_source(shape, device='cpu', source_df=None, source_scales=None, level_dims=None):
    """Sample from source distribution with optional per-level Student-t and scaling.
    source_df: scalar df → Student-t for all dims; list → per-level (None/0 = Gaussian, float = Student-t)
    source_scales: list of per-level scale factors applied after sampling
    """
    if isinstance(source_df, (list, tuple)):
        # Per-level: each slice sampled independently
        assert level_dims is not None, "level_dims required for per-level source_df"
        batch = shape[:-1]
        y = torch.empty(*shape, device=device)
        offset = 0
        for df, d in zip(source_df, level_dims):
            sl = (*batch, d)
            if df:
                normal = torch.randn(*sl, device=device)
                gamma  = torch._standard_gamma(torch.full(sl, df/2, device=device)) / (df/2)
                y[..., offset:offset+d] = normal / gamma.sqrt()
            else:
                y[..., offset:offset+d] = torch.randn(*sl, device=device)
            offset += d
    elif source_df:
        # Scalar df → Student-t for all dims
        normal = torch.randn(*shape, device=device)
        gamma  = torch._standard_gamma(torch.full(shape, source_df/2, device=device)) / (source_df/2)
        y = normal / gamma.sqrt()
    else:
        y = torch.randn(*shape, device=device)
    if source_scales is not None and level_dims is not None:
        offset = 0
        for scale, d in zip(source_scales, level_dims):
            y[..., offset:offset+d] *= scale
            offset += d
    return y

In [ ]:
#| export
@torch.no_grad()
def generate_samples_conditional(coarse_model, fine_model, n_samples, coarse_dim,
                                 target_dims, device='cpu', n_steps=20,
                                 step_fn=rk4_step, warp_s=0.5,
                                 coarse_source_df=None, coarse_source_scales=None,
                                 coarse_level_dims=None, fine_source_scales=None):
    """Two-stage conditional sampler: coarse flow → fine conditional flow.

    Conditioning signal: x1_pred_coarse = x_t_coarse + (1-t) * v_coarse
    — the coarse model's predicted endpoint at each step.  This is more
    informative than the noisy state x_t_coarse alone (especially at small t
    where x_t is mostly noise), and is in the same space as the final coarse
    embeddings, normalising out the t-dependence of the velocity scale.

    Both models share the *same* timestep grid — enforced by construction.
    The coarse model is called once per step to get v_coarse for x1_pred,
    then step_fn (which may call it again internally for RK4) advances the
    coarse state.  The fine model uses a simple Euler step conditioned on
    x1_pred_coarse computed at the start of each step.

    Training counterpart: at each batch, freeze coarse model, compute
    x1_pred_coarse = x_t_coarse + (1-t)*coarse_model(x_t_coarse, t),
    then train fine_model(x_t_fine, t, x1_pred_coarse) with flow-matching loss.

    Args:
        coarse_model:        first-stage flow (CrossLevelFlowModel / PerLevelFlowModel)
        fine_model:          ConditionalFineFlowModel
        coarse_dim:          total dim of coarse-level output (sum of PCA dims L0-L3)
        target_dims:         list of flattened dims for fine levels (e.g. [D_L4, D_L5])
        coarse_source_*:     source distribution kwargs forwarded to the coarse stage
        fine_source_scales:  optional per-level scale factors for fine-level noise
    Returns:
        coarse_out : [n_samples, coarse_dim]
        fine_out   : [n_samples, sum(target_dims)]
    """
    fine_dim = sum(target_dims)
    y_coarse = sample_source((n_samples, coarse_dim), device=device,
                             source_df=coarse_source_df,
                             source_scales=coarse_source_scales,
                             level_dims=coarse_level_dims)
    y_fine = sample_source((n_samples, fine_dim), device=device,
                           source_scales=fine_source_scales,
                           level_dims=target_dims)
    ts = warp_time(torch.linspace(0, 1, n_steps + 1), s=warp_s)
    coarse_model.eval(); fine_model.eval()
    for i in range(n_steps):
        dt   = (ts[i+1] - ts[i]).item()
        t_s  = ts[i].item()
        t    = torch.full((n_samples, 1), t_s, device=device)

        # x1_pred_coarse: coarse model's predicted endpoint at current state/time
        v_coarse_pred  = coarse_model(y_coarse, t)
        x1_pred_coarse = y_coarse + (1 - t_s) * v_coarse_pred  # [n_samples, coarse_dim]

        # Advance coarse state (step_fn may call coarse_model again internally for RK4)
        y_coarse = step_fn(coarse_model, y_coarse, t_s, dt)

        # Fine model conditioned on x1_pred_coarse; Euler step
        v_fine = fine_model(y_fine, t, x1_pred_coarse)
        y_fine = y_fine + v_fine * dt

    return y_coarse, y_fine

In [ ]:
#| eval: false
# Smoke test: ConditionalFineFlowModel forward pass
import torch
cond_dims, target_dims = [20, 40, 80, 160], [32, 16]
m = ConditionalFineFlowModel(cond_dims, target_dims, h_dim=64, n_layers=3, n_attn_layers=2, n_heads=4)
print(f'Params: {sum(p.numel() for p in m.parameters()):,}')
B = 4
v = m(torch.randn(B, sum(target_dims)), torch.rand(B), torch.randn(B, sum(cond_dims)))
assert v.shape == (B, sum(target_dims)), f"Expected {(B, sum(target_dims))}, got {v.shape}"
print(f'Output shape: {v.shape}  OK')

In [ ]:
#| export
def ann_repair(source, target, n_projections=1, chunk_size=None):
    """Approximate nearest-neighbor re-pairing of source and target batches.

    Sorts both source and target by their projection onto random unit vectors
    and pairs by rank — equivalent to exact 1-D OT along that direction.
    Fully on-device (GPU-friendly), O(B log B) per projection.

    chunk_size: if set, processes the batch in chunks of this size and repairs
    independently within each chunk. Smaller chunks are faster but less optimal;
    default (None) processes the whole batch at once.

    With n_projections > 1, tries multiple random directions and keeps the
    pairing with the lowest total squared transport cost.

    Args:
        source:        (B, D) tensor on any device
        target:        (B, D) tensor on same device
        n_projections: number of random projections to try per chunk
        chunk_size:    chunk size for within-batch processing (None = full batch)

    Returns:
        (source_repaired, target_repaired): re-ordered so source[i] ↔ target[i]
        approximately minimises total squared transport cost.
    """
    B, D = source.shape
    C = B if (chunk_size is None or chunk_size >= B) else chunk_size

    s_out = torch.empty_like(source)
    t_out = torch.empty_like(target)
    for start in range(0, B, C):
        end  = min(start + C, B)
        s, t = source[start:end], target[start:end]
        best_s, best_t, best_cost = s, t, float('inf')
        for _ in range(n_projections):
            proj   = torch.randn(D, device=s.device, dtype=s.dtype)
            proj   = proj / proj.norm()
            s_rep  = s[(s @ proj).argsort()]
            t_rep  = t[(t @ proj).argsort()]
            cost   = (s_rep - t_rep).pow(2).sum().item()
            if cost < best_cost:
                best_cost = cost
                best_s, best_t = s_rep, t_rep
        s_out[start:end] = best_s
        t_out[start:end] = best_t
    return s_out, t_out


In [ ]:
#| export
@torch.no_grad()
def generate_samples(model, n_samples, dim, device='cpu',
                     n_steps=20, step_fn=rk4_step, warp_s=0.5, source_df=None,
                     source_scales=None, level_dims=None):
    """Sample from the flow model: integrate noise → embedding space."""
    y = sample_source((n_samples, dim), device=device, source_df=source_df,
                      source_scales=source_scales, level_dims=level_dims)
    ts = torch.linspace(0, 1, n_steps + 1)
    ts = warp_time(ts, s=warp_s)
    model.eval()
    for i in range(n_steps):
        dt = (ts[i+1] - ts[i]).item()
        y  = step_fn(model, y, ts[i].item(), dt)
    return y

In [ ]:
#| export
def mmd_rbf(x, y, n_sub=2000):
    """Unbiased MMD² with RBF kernel, median bandwidth heuristic.
    x, y: (N, D) tensors. Subsamples to n_sub for speed."""
    if x.size(0) > n_sub: x = x[torch.randperm(x.size(0))[:n_sub]]
    if y.size(0) > n_sub: y = y[torch.randperm(y.size(0))[:n_sub]]
    xy = torch.cat([x, y], dim=0)
    sigma2 = torch.cdist(xy, xy).median().pow(2).clamp(min=1e-6)
    def rbf(a, b): return torch.exp(-torch.cdist(a, b).pow(2) / (2 * sigma2))
    return (rbf(x, x).mean() + rbf(y, y).mean() - 2 * rbf(x, y).mean()).item()


In [ ]:

#| export
def wasserstein_score(x, y, n_projections=200, n_sub=2000):
    """Sliced Wasserstein distance: average 1-D Wasserstein over random projections.
    Falls back gracefully if geomloss is unavailable.
    Returns nan on numerical failure (overflow, diverged samples, etc.).
    x, y: (N, D) numpy arrays."""
    try:
        import geomloss
        loss = geomloss.SamplesLoss("sinkhorn", p=2, blur=0.05)
        xt = torch.tensor(x[:n_sub]).float()
        yt = torch.tensor(y[:n_sub]).float()
        return loss(xt, yt).item()
    except ImportError:
        pass
    except Exception:
        return float('nan')
    try:
        from scipy.stats import wasserstein_distance
        rng = np.random.default_rng(0)
        D = x.shape[1]
        projs = rng.standard_normal((D, n_projections))
        projs /= np.linalg.norm(projs, axis=0, keepdims=True)
        px, py = x[:n_sub] @ projs, y[:n_sub] @ projs
        return float(np.mean([wasserstein_distance(px[:, i], py[:, i]) for i in range(n_projections)]))
    except Exception:
        return float('nan')


In [ ]:

#| export
@torch.no_grad()
def eval_flow(model, real_embeddings, n_samples=10000, n_steps=20, warp_s=0.5, device='cpu',
              source_df=None, source_scales=None, level_dims=None, gen=None, level_names=None):
    """Compare distributional statistics of real vs generated embeddings, per level.
    Returns flat dict with keys like 'L0/mmd', 'L0/wasserstein', 'L0/real_std', etc.
    Also returns global 'mmd' and 'wasserstein' for backward compatibility.
    real_embeddings: (N, D) tensor.
    gen: optional pre-computed generated samples (N, D) tensor — skips generate_samples().
    level_names: optional list of strings e.g. ['L4', 'L5'] to override default 'L0', 'L1' keys.
    """
    from scipy.stats import skew, kurtosis
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float()
    if gen is None:
        dim = real_embeddings.shape[1]
        gen = generate_samples(model, n_samples, dim, device=device,
                               n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                               source_scales=source_scales, level_dims=level_dims).cpu()
    else:
        gen = gen[:n_samples].float().cpu()
    r, g = real.numpy(), gen.numpy()

    metrics = {}
    # Global stats
    metrics['real_mean']  = float(r.mean())
    metrics['real_std']   = float(r.std())
    metrics['real_skew']  = float(skew(r.ravel()))
    metrics['real_kurt']  = float(kurtosis(r.ravel()))
    metrics['gen_mean']   = float(g.mean())
    metrics['gen_std']    = float(g.std())
    metrics['gen_skew']   = float(skew(g.ravel()))
    metrics['gen_kurt']   = float(kurtosis(g.ravel()))
    metrics['mmd']        = mmd_rbf(real, gen)
    metrics['wasserstein'] = wasserstein_score(r, g)

    # Per-level stats
    if level_dims is not None:
        offset = 0
        for i, d in enumerate(level_dims):
            rl = r[:, offset:offset+d]
            gl = g[:, offset:offset+d]
            rt, gt = torch.tensor(rl), torch.tensor(gl)
            lname = level_names[i] if level_names else f'L{i}'
            metrics[f'{lname}/real_std']    = float(rl.std())
            metrics[f'{lname}/gen_std']     = float(gl.std())
            metrics[f'{lname}/real_kurt']   = float(kurtosis(rl.ravel()))
            metrics[f'{lname}/gen_kurt']    = float(kurtosis(gl.ravel()))
            metrics[f'{lname}/mmd']         = mmd_rbf(rt, gt)
            metrics[f'{lname}/wasserstein'] = wasserstein_score(rl, gl)
            offset += d

    w = max(len(k) for k in metrics)
    for k, v in metrics.items():
        print(f'  {k:{w}s} = {v:.4f}')
    return metrics

In [ ]:
#| export
def eval_jacobian_norm_vs_t(model, x_sample, n_t=20, n_epsilon=4, cond=None, device='cpu'):
    """Estimate Frobenius norm of the Jacobian dv/dx as a function of t.

    Uses Hutchinson estimator: E_ε[||J^T ε||²] = ||J||²_F  with Rademacher ε.
    A peak in the curve at some t* reveals where the flow is making its hardest
    topological decisions (routing mass to disjoint clusters).

    Args:
        model:     velocity field; called as model(x, t) or model(x, t, cond)
        x_sample:  (B, D) batch of real data points
        n_t:       number of time steps to sweep
        n_epsilon: number of Rademacher samples per t (more = lower variance)
        cond:      optional conditioning tensor (B, cond_dim) for conditional models
        device:    compute device
    Returns:
        t_vals (list[float]), norm_vals (list[float])
    """
    model.eval()
    x_sample = x_sample.to(device).float()
    t_vals, norm_vals = [], []

    for t_val in torch.linspace(0.01, 0.99, n_t).tolist():
        t = torch.full((len(x_sample), 1), t_val, device=device)
        x = x_sample.detach().requires_grad_(True)

        est_list = []
        for _ in range(n_epsilon):
            epsilon = (torch.randint_like(x, low=0, high=2).float() * 2 - 1)  # Rademacher ±1
            if cond is not None:
                v = model(x, t, cond.to(device))
            else:
                v = model(x, t)
            vjp = torch.autograd.grad(v, x, grad_outputs=epsilon,
                                       retain_graph=True, create_graph=False)[0]
            # Frobenius norm estimator: E[||J^T ε||²] = ||J||²_F
            est_list.append(vjp.pow(2).sum(dim=-1))   # (B,)

        norm_sq = torch.stack(est_list).mean(0).mean().item()   # scalar
        t_vals.append(t_val)
        norm_vals.append(norm_sq ** 0.5)              # sqrt for Frobenius norm

    return t_vals, norm_vals

In [ ]:

#| export
@torch.no_grad()
def plot_level_histograms(model, real_embeddings, level_dims, n_samples=10000,
                          n_steps=20, warp_s=0.5, device='cpu', n_bins=100, source_df=None,
                          source_scales=None, epoch=None, gen=None, level_names=None):
    """Return dict of per-level histogram figures {'L0': fig, 'L1': fig, ...}.
    level_dims: list of ints, flattened PCA dims per level e.g. [20, 80, 320, 1280]
    real_embeddings: (N, sum(level_dims)) tensor
    gen: optional pre-computed generated samples — skips generate_samples().
    level_names: optional list of strings e.g. ['L4', 'L5'] to override default 'L0', 'L1' labels.
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float().numpy()
    if gen is None:
        dim = real_embeddings.shape[1]
        gen = generate_samples(model, n_samples, dim, device=device,
                               n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                               source_scales=source_scales, level_dims=level_dims).cpu().numpy()
    else:
        gen = gen[:n_samples].float().cpu().numpy()
    figs = {}
    offset = 0
    for i, d in enumerate(level_dims):
        r = real[:, offset:offset+d].ravel()
        g = gen[:,  offset:offset+d].ravel()
        lim = np.percentile(np.abs(np.concatenate([r, g])), 99)
        bins = np.linspace(-lim, lim, n_bins + 1)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(r, bins=bins, alpha=0.5, color='steelblue', label='real', density=True)
        ax.hist(g, bins=bins, alpha=0.5, color='darkorange', label='gen',  density=True)
        lname = level_names[i] if level_names else f'L{i}'
        title = f'{lname} ({d}d)'
        if epoch is not None: title += f' — Epoch {epoch}'
        ax.set_title(title)
        ax.set_xlabel('value')
        ax.legend(fontsize=8)
        plt.tight_layout()
        figs[lname] = fig
        offset += d
    return figs

In [ ]:
#| export
@torch.no_grad()
def plot_level_scatter(model, real_embeddings, level_dims, n_samples=5000,
                       n_steps=20, warp_s=0.5, device='cpu',
                       source_df=None, source_scales=None, epoch=None,
                       gen=None, level_names=None):
    """Return dict of per-level 3D PCA scatter plots {'L0/real': fig, 'L0/gen': fig, ...}.
    gen: optional pre-computed generated samples — skips generate_samples().
    level_names: optional list of strings e.g. ['L4', 'L5'] to override default 'L0', 'L1' labels.
    """
    from midi_rae.viz import pca_project, plot_embeddings_3d
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float()
    if gen is None:
        dim = real_embeddings.shape[1]
        gen = generate_samples(model, n_samples, dim, device=device,
                               n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                               source_scales=source_scales, level_dims=level_dims).cpu()
    else:
        gen = gen[:n_samples].float().cpu()
    figs = {}
    offset = 0
    for i, d in enumerate(level_dims):
        r = real[:, offset:offset+d]
        g = gen[:,  offset:offset+d]
        lname = level_names[i] if level_names else f'L{i}'
        title_sfx = f' — Epoch {epoch}' if epoch is not None else ''
        r3 = pca_project(r)
        g3 = pca_project(g)
        if r3 is not None: figs[f'{lname}/real'] = plot_embeddings_3d(r3, color_by='random', title=f'{lname} ({d}d) real{title_sfx}')
        if g3 is not None: figs[f'{lname}/gen']  = plot_embeddings_3d(g3, color_by='random', title=f'{lname} ({d}d) gen{title_sfx}')
        offset += d
    return figs

In [ ]:
#| export
@torch.no_grad()
def decode_flow_to_piano_rolls(coarse_pca, fine_emb, pca_models,
                                coarse_level_dims, fine_level_dims, fine_levels_idx,
                                cfg, decoder, device, n_samples=16):
    """Decode flow-generated coarse PCA + fine embeddings directly to piano rolls (no HMEP).
    coarse_pca:  (B, sum_coarse_dims) PCA-compressed coarse embeddings
    fine_emb:    (B, sum_fine_dims) raw fine-level embeddings from conditional flow
    fine_levels_idx: list of level indices e.g. [4, 5]
    """
    from midi_rae.generate import build_patch_states, batch_patch_states, build_enc_out, make_grid_pos, binarize
    from midi_rae.core import PatchState

    B = min(n_samples, coarse_pca.shape[0])
    coarse_pca = coarse_pca[:B].float()
    fine_emb   = fine_emb[:B].float()

    # Inverse PCA → coarse PatchState list
    states = [build_patch_states(coarse_pca[b], pca_models, coarse_level_dims, device)
              for b in range(B)]
    all_levels = batch_patch_states(states)

    # Fine levels: reshape directly from flat flow output (no HMEP)
    n_stages  = len(list(cfg.model.depths))
    embed_dim = cfg.model.embed_dim
    enc_dims  = [int(embed_dim * 2**(n_stages - 1 - i)) for i in range(n_stages)]
    offset = 0
    for j, li in enumerate(fine_levels_idx):
        d             = fine_level_dims[j]
        n_patches     = 4 ** li
        dim_per_patch = enc_dims[li]
        emb = fine_emb[:, offset:offset+d].reshape(B, n_patches, dim_per_patch).to(device)
        pos = make_grid_pos(n_patches, device)
        all_levels.append(PatchState(emb=emb, pos=pos,
                                     non_empty=torch.ones(B, n_patches, device=device),
                                     mae_mask=torch.ones(n_patches, device=device)))
        offset += d

    enc_out = build_enc_out(all_levels)
    recons  = decoder(enc_out)
    return binarize(recons)

In [ ]:
#| export
def train_flow(model, dataset, n_epochs=100, lr=3e-4, batch_size=2048,
               warp_s=0.5, device='cpu', checkpoint_dir=None, save_every=10,
               eval_every=10, viz_every=50, use_wandb=False, steps_per_epoch=None,
               source_df=None, source_scales=None, checkpoint=None, cfg=None,
               lr_restart_epochs=500, lr_warmup_frac=0.15, grad_clip=1.0,
               repair_every=1, n_repair_projections=1, repair_chunk_size=None,
               ema_eta=0.97, ema_start_epoch=100):
    """Train flow matching model on embedding dataset.

    Source: N(0,I) sampled fresh each step.
    Target: embeddings from dataset.
    Loss:   MSE between predicted and true (constant) velocity.
    checkpoint: path to a saved checkpoint to resume from (optional).
    cfg: config dict/object passed to save_checkpoint (optional).
    lr_restart_epochs: T_0 (first cycle length) for warm-restart schedule.
    lr_warmup_frac: fraction of each cycle spent on linear warmup to peak lr (default 0.15).
    viz_every: how often (epochs) to log histograms + scatter plots to W&B.
    grad_clip: max norm for gradient clipping (0 = disabled).
    repair_every: re-pair source/target every N batches via ann_repair (0 = disabled).
    n_repair_projections: random projections per ann_repair call (more = better, slower).
    repair_chunk_size: chunk size for ann_repair (None = full batch).
    ema_eta: EMA decay rate (updated every batch).
    ema_start_epoch: epoch at which eval/sampling switches to EMA model.
    """
    from midi_rae.utils import save_checkpoint, load_checkpoint, EMAModel
    self_cond = getattr(model, 'self_condition', False)
    model = model.to(device)
    ema_model = EMAModel(model, eta=ema_eta, update_every=1, dtype=torch.float32)
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                    num_workers=2, pin_memory=(device != 'cpu'), drop_last=True)
    dl_iter = None
    _steps = steps_per_epoch or len(dl)
    if use_wandb:
        import wandb
        wandb.config.update(dict(n_epochs=n_epochs, lr=lr, batch_size=batch_size,
                                 warp_s=warp_s, dim=dataset.embeddings.shape[1],
                                 self_condition=self_cond), allow_val_change=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    global_step = 0
    epoch_start = 0
    real_scatter_logged = False

    if checkpoint:
        model, ckpt = load_checkpoint(model, checkpoint, return_all=True)
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        epoch_start = ckpt['epoch']
        global_step = epoch_start * _steps
        print(f"Resumed from {checkpoint} (epoch {epoch_start})")
        ema_model.ema.load_state_dict(model.state_dict())

    scheduler = make_warmup_cosine_restart_scheduler(
        optimizer, T_0=lr_restart_epochs, T_mult=2, warmup_frac=lr_warmup_frac, eta_min=1e-6)
    for _ in range(epoch_start): scheduler.step()  # fast-forward if resuming

    for epoch in range(epoch_start, n_epochs):
        model.train()
        epoch_loss = 0.
        if steps_per_epoch:
            if dl_iter is None:
                import itertools; dl_iter = itertools.cycle(dl)
            batches = (next(dl_iter) for _ in range(_steps))
        else:
            batches = dl
        pbar = tqdm(batches, total=_steps, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False)
        for target in pbar:
            target = target.to(device)
            B, D   = target.shape
            source = sample_source((B, D), device=device, source_df=source_df,
                                   source_scales=source_scales,
                                   level_dims=getattr(dataset, 'level_dims', None))
            if repair_every and global_step % repair_every == 0:
                source, target = ann_repair(source, target,
                                            n_projections=n_repair_projections,
                                            chunk_size=repair_chunk_size)

            t = torch.rand(B, 1, device=device)
            if warp_s != 1.0:
                t = warp_time(t, s=warp_s)

            x_t = (1 - t) * source + t * target
            v   = target - source

            # Self-conditioning: 50% of batches, do a first pass to get x̂₁,
            # then use it as conditioning for the actual training pass.
            x_self_cond = None
            if self_cond and torch.rand(1).item() < 0.5:
                with torch.no_grad():
                    v_first = model(x_t, t, None)
                    x_self_cond = (x_t + (1 - t) * v_first).detach()

            optimizer.zero_grad()
            v_pred = model(x_t, t, x_self_cond)
            loss   = loss_fn(v_pred, v)
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            optimizer.step()
            ema_model.update(model)

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.4f}')
            global_step += 1

        scheduler.step()
        avg_loss = epoch_loss / len(dl)
        cur_lr = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch+1}/{n_epochs}  loss={avg_loss:.4f}  lr={cur_lr:.2e}')
        if use_wandb: wandb.log({'train/epoch_loss': avg_loss, 'train/lr': cur_lr, 'epoch': epoch+1}, step=global_step)

        if eval_every and (epoch + 1) % eval_every == 0:
            print(f'  --- eval epoch {epoch+1} ---')
            eval_model = ema_model.ema if (epoch + 1) >= ema_start_epoch else model
            metrics = eval_flow(eval_model, dataset.embeddings, device=device, warp_s=warp_s,
                               source_df=source_df, source_scales=source_scales,
                               level_dims=getattr(dataset, 'level_dims', None))
            if use_wandb:
                import wandb
                log_dict = {f'eval/{k}': v for k, v in metrics.items()}
                if hasattr(dataset, 'level_dims') and viz_every and (epoch + 1) % viz_every == 0:
                    figs = plot_level_histograms(eval_model, dataset.embeddings,
                                               dataset.level_dims, device=device, warp_s=warp_s,
                                               source_df=source_df, source_scales=source_scales,
                                               epoch=epoch+1)
                    import matplotlib.pyplot as plt
                    for lname, fig in figs.items():
                        log_dict[f'media/hist_{lname}'] = wandb.Image(fig, caption=f'Epoch {epoch+1}')
                        plt.close(fig)
                    del figs
                    scatters = plot_level_scatter(eval_model, dataset.embeddings,
                                                 dataset.level_dims, device=device, warp_s=warp_s,
                                                 source_df=source_df, source_scales=source_scales,
                                                 epoch=epoch+1)
                    for lname, fig in scatters.items():
                        is_real = lname.endswith('/real')
                        if is_real and real_scatter_logged:
                            fig.data = []
                            continue
                        log_dict[f'media/scatter_{lname.replace("/", "_")}'] = wandb.Html(fig.to_html())
                    real_scatter_logged = True
                    del scatters
                    gc.collect()
                log_dict['epoch'] = epoch+1
                wandb.log(log_dict, step=global_step)
            model.train()

        save_checkpoint([model, ema_model.ema], epoch+1, avg_loss, cfg or {}, optimizer=optimizer,
                        save_every=save_every, tag=f'flow_{model.__class__.__name__}')

    print(f"FINISHED. Best metric: final loss={avg_loss:.4f}")
    return model


In [ ]:
#| export
def make_warmup_cosine_restart_scheduler(optimizer, T_0, T_mult=2, warmup_frac=0.15, eta_min=1e-6):
    """LambdaLR implementing true warm restarts: linear ramp-up → cosine decay per cycle.
    Periods double each restart (T_mult=2): T_0, T_0*2, T_0*4, ...
    warmup_frac: fraction of each cycle spent warming up to peak lr.
    eta_min: minimum lr (as absolute value, not a multiplier)."""
    base_lr = optimizer.param_groups[0]['lr']

    def get_cycle(epoch):
        """Return (cycle index, position within cycle, cycle length)."""
        T_i, T_prev = T_0, 0
        while T_prev + T_i <= epoch:
            T_prev += T_i
            T_i = int(T_i * T_mult)
        return T_prev, T_i   # cycle_start, cycle_length

    def lr_lambda(epoch):
        cycle_start, T_i = get_cycle(epoch)
        T_cur = epoch - cycle_start
        warmup_end = max(1, int(T_i * warmup_frac))
        if T_cur < warmup_end:
            return T_cur / warmup_end                              # linear warmup → 1.0 (base_lr)
        progress = (T_cur - warmup_end) / max(1, T_i - warmup_end)
        cos_val = 0.5 * (1 + math.cos(math.pi * progress))        # 1 → 0
        return eta_min / base_lr + (1 - eta_min / base_lr) * cos_val

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


In [ ]:
#| export
def train_flow_conditional(coarse_model, fine_model, dataset,
                           n_epochs=100000, lr=1e-3, batch_size=4096,
                           warp_s=0.5, device='cpu',
                           checkpoint_dir=None, save_every=10, eval_every=10, viz_every=20,
                           use_wandb=False, steps_per_epoch=None,
                           fine_source_scales=None, checkpoint=None, cfg=None,
                           lr_restart_epochs=500, lr_warmup_frac=0.15, grad_clip=1.0,
                           ema_eta=0.97, ema_start_epoch=100, fine_level_names=None,
                           decoder=None, pca_models=None, fine_levels_idx=None):
    """Train ConditionalFineFlowModel with a frozen coarse model as conditioning.

    decoder, pca_models, fine_levels_idx: if all provided, decode piano rolls during viz.
    fine_level_names: optional list of strings e.g. ['L4', 'L5'] for metric/histogram labels.
    viz_every: how often (epochs) to log histograms + scatter plots to W&B.
    Checkpoints save both live model (with optimizer) and EMA model under distinct tags.
    Resume by pointing ++checkpoint at the EMA _best.pt file.
    """
    from midi_rae.utils import save_checkpoint, load_checkpoint, EMAModel
    from midi_rae.data import ChunkShuffleSampler, ConditionalFlowDataset
    coarse_model = coarse_model.to(device)
    coarse_model.eval()
    for p in coarse_model.parameters(): p.requires_grad_(False)

    fine_model = fine_model.to(device)
    ema_model  = EMAModel(fine_model, eta=ema_eta, update_every=1, dtype=torch.float32)

    # Use ChunkShuffleSampler for lazy datasets to avoid chunk thrashing;
    # fall back to shuffle=True for fully-loaded datasets.
    if isinstance(dataset, ConditionalFlowDataset):
        sampler = ChunkShuffleSampler(dataset)
        dl = DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                        num_workers=0, pin_memory=(device != 'cpu'), drop_last=True)
    else:
        dl = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        num_workers=2, pin_memory=(device != 'cpu'), drop_last=True)
    dl_iter = None
    _steps  = steps_per_epoch or len(dl)

    optimizer = optim.Adam(fine_model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    scheduler = make_warmup_cosine_restart_scheduler(
        optimizer, T_0=lr_restart_epochs, T_mult=2, warmup_frac=lr_warmup_frac, eta_min=1e-6)

    global_step = 0
    epoch_start = 0
    real_scatter_logged = False

    if checkpoint:
        # checkpoint points at EMA file; restore both live model and EMA from it
        fine_model, ckpt = load_checkpoint(fine_model, checkpoint, return_all=True)
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        epoch_start = ckpt['epoch']
        global_step = epoch_start * _steps
        for _ in range(epoch_start): scheduler.step()
        ema_model.ema.load_state_dict(fine_model.state_dict())
        print(f"Resumed from {checkpoint} (epoch {epoch_start})")

    fine_level_dims   = dataset.fine_level_dims
    coarse_dim        = dataset.coarse.shape[1]
    coarse_level_dims = dataset.coarse_level_dims

    # Pre-load eval fine data from the first chunk (avoids scanning full dataset)
    def _get_eval_fine(n=2000):
        dataset._load_fine_chunk(0)
        return dataset._fine_chunk_data[:n].float()

    for epoch in range(epoch_start, n_epochs):
        fine_model.train()
        epoch_loss = 0.
        if steps_per_epoch:
            if dl_iter is None:
                import itertools; dl_iter = itertools.cycle(dl)
            batches = (next(dl_iter) for _ in range(_steps))
        else:
            batches = dl
        pbar = tqdm(batches, total=_steps, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False)
        for real_coarse, real_fine in pbar:
            real_coarse = real_coarse.to(device)
            real_fine   = real_fine.to(device)
            B = real_coarse.size(0)

            t = warp_time(torch.rand(B, 1, device=device), s=warp_s)
            noise_coarse = torch.randn_like(real_coarse)
            noise_fine   = sample_source((B, real_fine.size(1)), device=device,
                                         source_scales=fine_source_scales,
                                         level_dims=fine_level_dims)

            x_t_coarse = (1 - t) * noise_coarse + t * real_coarse
            x_t_fine   = (1 - t) * noise_fine   + t * real_fine
            v_fine_target = real_fine - noise_fine

            with torch.no_grad():
                v_coarse       = coarse_model(x_t_coarse, t)
                x1_pred_coarse = x_t_coarse + (1 - t) * v_coarse

            optimizer.zero_grad()
            v_pred = fine_model(x_t_fine, t, x1_pred_coarse)
            loss   = loss_fn(v_pred, v_fine_target)
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(fine_model.parameters(), max_norm=grad_clip)
            optimizer.step()
            ema_model.update(fine_model)

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.4f}')
            global_step += 1

        scheduler.step()
        avg_loss = epoch_loss / _steps
        cur_lr   = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch+1}/{n_epochs}  loss={avg_loss:.4f}  lr={cur_lr:.2e}')

        log_dict = {'train/loss': avg_loss, 'train/lr': cur_lr, 'epoch': epoch+1}

        if eval_every and (epoch + 1) % eval_every == 0:
            eval_model = ema_model.ema if (epoch + 1) >= ema_start_epoch else fine_model
            gen_coarse, gen_fine = generate_samples_conditional(
                coarse_model, eval_model,
                n_samples=2000, coarse_dim=coarse_dim,
                target_dims=fine_level_dims, device=device,
                n_steps=20, warp_s=warp_s,
                coarse_level_dims=coarse_level_dims,
                fine_source_scales=fine_source_scales,
            )
            real_fine_eval = _get_eval_fine(2000)
            gen_fine_cpu = gen_fine.cpu()

            metrics = eval_flow(eval_model, real_fine_eval,
                                n_samples=2000, level_dims=fine_level_dims,
                                gen=gen_fine_cpu, level_names=fine_level_names)
            log_dict.update({f'eval/{k}': v for k, v in metrics.items()})

            if use_wandb and viz_every and (epoch + 1) % viz_every == 0:
                import wandb
                import matplotlib
                matplotlib.use('Agg')
                import matplotlib.pyplot as plt
                from torchvision.utils import make_grid

                figs = plot_level_histograms(eval_model, real_fine_eval,
                                             fine_level_dims, epoch=epoch+1,
                                             gen=gen_fine_cpu, level_names=fine_level_names)
                for lname, fig in figs.items():
                    log_dict[f'media/hist_{lname}'] = wandb.Image(fig, caption=f'Epoch {epoch+1}')
                    plt.close(fig)
                del figs

                scatters = plot_level_scatter(eval_model, real_fine_eval,
                                              fine_level_dims, epoch=epoch+1,
                                              gen=gen_fine_cpu, level_names=fine_level_names)
                for lname, fig in scatters.items():
                    is_real = lname.endswith('/real')
                    if is_real and real_scatter_logged:
                        fig.data = []
                        continue
                    log_dict[f'media/scatter_{lname.replace("/", "_")}'] = wandb.Html(fig.to_html())
                real_scatter_logged = True
                del scatters

                if decoder is not None and pca_models is not None and fine_levels_idx is not None:
                    rolls = decode_flow_to_piano_rolls(
                        gen_coarse.cpu(), gen_fine_cpu,
                        pca_models, coarse_level_dims, fine_level_dims,
                        fine_levels_idx, cfg, decoder, device, n_samples=16)
                    grid = make_grid(rolls[:16], nrow=4, normalize=True)
                    log_dict['media/piano_rolls_gen'] = wandb.Image(grid, caption=f'Epoch {epoch+1}')

                # Jacobian norm vs t — reveals where the flow makes hard topological decisions
                real_coarse_eval = dataset.coarse[:len(real_fine_eval)].to(device)
                with torch.no_grad():
                    cond_eval = coarse_model(real_coarse_eval,
                                             torch.ones(len(real_coarse_eval), 1, device=device) * 0.5)
                t_vals, jac_norms = eval_jacobian_norm_vs_t(
                    eval_model, real_fine_eval[:256], n_t=20, n_epsilon=4,
                    cond=cond_eval[:256], device=device)
                jac_table = wandb.Table(columns=['t', 'jacobian_norm'],
                                        data=[[t, n] for t, n in zip(t_vals, jac_norms)])
                log_dict['eval/jacobian_norm_vs_t'] = wandb.plot.line(
                    jac_table, 't', 'jacobian_norm', title='Jacobian Frobenius Norm vs t')

                gc.collect()

            fine_model.train()

        if use_wandb:
            import wandb
            wandb.log(log_dict, step=global_step)

        if save_every and (epoch + 1) % save_every == 0:
            # Save live model (with optimizer) and EMA model separately under distinct tags
            # so both can be restored independently on resume.
            save_checkpoint(fine_model,    epoch+1, avg_loss, cfg or {},
                            optimizer=optimizer, save_every=save_every,
                            tag='flow2_fine_live')
            save_checkpoint(ema_model.ema, epoch+1, avg_loss, cfg or {},
                            save_every=save_every, tag='flow2_fine_ema')

In [ ]:
#| export
#| eval: false
import hydra
from omegaconf import DictConfig

def _run_flow2(cfg: DictConfig):
    """Second-stage conditional fine-level flow training (called from train_flow_main)."""
    from midi_rae.data import ConditionalFlowDataset
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"device = {device}")

    flow2 = cfg.flow2
    fine_levels = list(flow2.get('fine_levels', [4, 5]))
    dataset = ConditionalFlowDataset(
        pca_dir     = flow2.pca_dir,
        encoded_dir = flow2.encoded_dir,
        pca_levels  = list(flow2.get('pca_levels',  ['L0','L1','L2','L3'])),
        fine_levels = fine_levels,
        emb_key     = flow2.get('emb_key', 'emb1'),
    )
    print(f"  {len(dataset)} samples  "
          f"coarse_dims={dataset.coarse_level_dims}  fine_dims={dataset.fine_level_dims}")

    coarse_level_dims = dataset.coarse_level_dims
    coarse_model = CrossLevelFlowModel(
        level_dims   = coarse_level_dims,
        h_dim        = cfg.flow.h_dim,
        n_layers     = cfg.flow.n_layers,
        n_attn_layers= cfg.flow.get('n_attn_layers', 2),
        n_heads      = cfg.flow.get('n_heads', 8),
        t_dim        = cfg.flow.get('t_dim', 64),
    )
    from midi_rae.utils import load_checkpoint
    coarse_ckpt = flow2.get('coarse_ckpt', None)
    if coarse_ckpt and str(coarse_ckpt).lower() not in ('none', 'null', ''):
        coarse_ckpt = os.path.expandvars(os.path.expanduser(str(coarse_ckpt)))
        coarse_model = load_checkpoint(coarse_model, coarse_ckpt)
        print(f"  Loaded coarse model from {coarse_ckpt}")
    else:
        print("  coarse_ckpt not set — coarse model starts from random weights")

    fine_model = ConditionalFineFlowModel(
        cond_dims    = coarse_level_dims,
        target_dims  = dataset.fine_level_dims,
        h_dim        = flow2.h_dim,
        n_layers     = flow2.n_layers,
        n_attn_layers= flow2.get('n_attn_layers', 2),
        n_heads      = flow2.get('n_heads', 8),
        t_dim        = cfg.flow.get('t_dim', 64),
    )
    n_params = sum(p.numel() for p in fine_model.parameters())
    print(f"  ConditionalFineFlowModel: {n_params:,} parameters")

    use_wandb = hasattr(cfg, 'wandb') and hasattr(cfg.wandb, 'flow_project')
    if use_wandb:
        import wandb
        wandb.init(project=cfg.wandb.flow_project, config=dict(flow2))
        wandb.define_metric("epoch")
        wandb.define_metric("*", step_metric="epoch")
        if hasattr(cfg, 'tag'): wandb.run.name = f"{cfg.tag}_{wandb.run.name}"

    fine_source_scales = list(flow2.get('fine_source_scales', [])) or None
    checkpoint = os.path.expandvars(os.path.expanduser(cfg.get('checkpoint', '') or '')) or None
    fine_level_names = [f'L{l}' for l in fine_levels]

    # Load decoder + PCA models for piano roll visualization (optional)
    decoder, pca_models = None, None
    decoder_ckpt = os.path.expandvars(os.path.expanduser(str(cfg.generate.get('decoder_ckpt', '') or '')))
    if decoder_ckpt:
        import pickle
        from pathlib import Path
        from midi_rae.swin import SwinDecoder
        m = cfg.model
        decoder = SwinDecoder(
            img_height=cfg.data.image_size, img_width=cfg.data.image_size,
            patch_h=m.patch_h, patch_w=m.patch_w, out_channels=cfg.data.in_channels,
            embed_dim=m.embed_dim, depths=list(m.dec_depths),
            num_heads=list(m.dec_num_heads), window_size=m.window_size,
            mlp_ratio=m.mlp_ratio, drop_path_rate=0.0)
        decoder = load_checkpoint(decoder, decoder_ckpt).to(device).eval()
        for p in decoder.parameters(): p.requires_grad_(False)
        pca_dir = Path(os.path.expandvars(os.path.expanduser(flow2.pca_dir)))
        n_coarse = len(coarse_level_dims)
        pca_models = {}
        for i in range(n_coarse):
            with open(pca_dir / f'pca_L{i}_n20.pkl', 'rb') as f:
                pca_models[i] = pickle.load(f)
        print(f"  Loaded decoder + {n_coarse} PCA models for piano roll viz")

    train_flow_conditional(
        coarse_model, fine_model, dataset,
        n_epochs          = flow2.n_epochs,
        lr                = flow2.lr,
        batch_size        = flow2.batch_size,
        warp_s            = cfg.flow.warp_s,
        device            = device,
        checkpoint_dir    = flow2.checkpoint_dir,
        save_every        = flow2.get('save_every', 10),
        eval_every        = flow2.get('eval_every', 10),
        viz_every         = flow2.get('viz_every', 20),
        use_wandb         = use_wandb,
        steps_per_epoch   = flow2.get('steps_per_epoch', None),
        fine_source_scales= fine_source_scales,
        checkpoint        = checkpoint,
        cfg               = cfg,
        lr_restart_epochs = flow2.get('lr_restart_epochs', 500),
        lr_warmup_frac    = flow2.get('lr_warmup_frac', 0.15),
        grad_clip         = flow2.get('grad_clip', 1.0),
        ema_eta           = flow2.get('ema_eta', 0.97),
        ema_start_epoch   = flow2.get('ema_start_epoch', 100),
        fine_level_names  = fine_level_names,
        decoder           = decoder,
        pca_models        = pca_models,
        fine_levels_idx   = fine_levels,
    )
    if use_wandb: wandb.finish()


@hydra.main(version_base=None, config_path="../configs", config_name="config_swin")
def train_flow_main(cfg: DictConfig):

    if cfg.get("flow_stage", 1) == 2:
        _run_flow2(cfg)
        return
    import glob as _glob
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"device = {device}")

    paths = sorted(_glob.glob(os.path.expandvars(os.path.expanduser(cfg.flow.embedding_glob))))
    assert paths, f"No files found matching {cfg.flow.embedding_glob}"
    print(f"Loading {len(paths)} file(s)...")
    source_scales = list(cfg.flow.source_scales) if cfg.flow.get("source_scales") else None
    raw_df = cfg.flow.get("source_df", None)
    source_df = list(raw_df) if hasattr(raw_df, '__iter__') else raw_df
    n_levels = len(source_scales) if source_scales else None
    levels = [f'L{i}' for i in range(n_levels)] if n_levels else None
    if source_df: source_df = source_df[:n_levels]
    dataset = EmbeddingDataset(paths, levels=levels)
    dim = dataset.embeddings.shape[1]
    print(f"  {len(dataset)} samples, dim={dim}, level_dims={dataset.level_dims}")

    self_condition = cfg.flow.get('self_condition', False)
    t_dim = cfg.flow.get('t_dim', 64)
    model_type = cfg.flow.get('model_type', 'per_level')
    if model_type == 'cross_level':
        model = CrossLevelFlowModel(level_dims=dataset.level_dims,
                                    h_dim=cfg.flow.h_dim, n_layers=cfg.flow.n_layers,
                                    n_attn_layers=cfg.flow.get('n_attn_layers', 2),
                                    n_heads=cfg.flow.get('n_heads', 8),
                                    self_condition=self_condition, t_dim=t_dim)
    else:
        model = PerLevelFlowModel(level_dims=dataset.level_dims,
                                  h_dim=cfg.flow.h_dim, n_layers=cfg.flow.n_layers,
                                  self_condition=self_condition, t_dim=t_dim)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {model.__class__.__name__}: {n_params:,} parameters (self_condition={self_condition}, t_dim={t_dim})")

    use_wandb = hasattr(cfg, "wandb") and hasattr(cfg.wandb, "flow_project")
    if use_wandb:
        import wandb
        wandb.init(project=cfg.wandb.flow_project, config=dict(cfg.flow))
        wandb.define_metric("epoch")
        wandb.define_metric("*", step_metric="epoch")
        if hasattr(cfg, "tag"): wandb.run.name = f"{cfg.tag}_{wandb.run.name}"

    checkpoint = os.path.expandvars(os.path.expanduser(cfg.get('checkpoint', '') or '')) or None

    train_flow(model, dataset,
               n_epochs=cfg.flow.n_epochs, lr=cfg.flow.lr, batch_size=cfg.flow.batch_size,
               warp_s=cfg.flow.warp_s, device=device,
               checkpoint_dir=cfg.flow.checkpoint_dir, save_every=cfg.flow.save_every,
               eval_every=cfg.flow.eval_every, viz_every=cfg.flow.get('viz_every', 25),
               use_wandb=use_wandb, steps_per_epoch=cfg.flow.steps_per_epoch,
               source_df=source_df, source_scales=source_scales,
               checkpoint=checkpoint, cfg=cfg,
               lr_restart_epochs=cfg.flow.get('lr_restart_epochs', 500),
               lr_warmup_frac=cfg.flow.get('lr_warmup_frac', 0.15),
               repair_every=cfg.flow.get('repair_every', 1),
               n_repair_projections=cfg.flow.get('n_repair_projections', 1),
               repair_chunk_size=cfg.flow.get('repair_chunk_size', None),
               ema_eta=cfg.flow.get('ema_eta', 0.97),
               ema_start_epoch=cfg.flow.get('ema_start_epoch', 100))

    if use_wandb: wandb.finish()

if __name__ == "__main__":
    train_flow_main()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()